# 01 · Exploración de la base — SEPA

Radiografía de la base de precios: **cobertura** (cadenas, provincias, sucursales, días),
**calidad** (faltantes, sentinelas, distribución de precios), **universo de cadenas** y
**mapa de sucursales**. Genera agregados livianos reutilizables por los demás notebooks.

> Requiere haber corrido `00_setup_datos.ipynb` (al menos el mes de muestra). Ver `docs/`.


## 1. Entorno y datos

In [ ]:
try:
    import google.colab  # noqa
    EN_COLAB = True
except ImportError:
    EN_COLAB = False
if EN_COLAB:
    !pip -q install duckdb pyarrow gdown pyyaml folium matplotlib

import sys, os
from pathlib import Path
REPO_URL = "https://github.com/santiagoriverti/precios_supermercados_argentina.git"
if EN_COLAB:
    if not Path("precios_supermercados_argentina").exists():
        !git clone -q $REPO_URL
    REPO = Path("precios_supermercados_argentina").resolve()
else:
    REPO = Path.cwd()
    while REPO.name and not (REPO / "src" / "precios_sepa").exists():
        REPO = REPO.parent
sys.path.insert(0, str(REPO / "src")); os.chdir(REPO)
import pandas as pd, numpy as np, duckdb
import precios_sepa
cfg = precios_sepa.load_settings()
print("Repo:", REPO)

In [ ]:
# Localizar el Parquet (del nb00 o del build local). Si no hay, construir una muestra capeada.
# En Colab cada notebook es una VM nueva: el Parquet del nb00 NO persiste, asi que este
# notebook es autosuficiente (ubica la base igual que el nb00 y arma su propia muestra).
from precios_sepa.io import descubrir_archivos
from precios_sepa.ingest import procesar_archivo

PARQUET = REPO / "data" / "interim" / "sepa"
glob = str(PARQUET / "**" / "*.parquet")

def _hay_parquet():
    return any(PARQUET.rglob("*.parquet"))

def _localizar_base():
    _local = Path(cfg["rutas"]["base_extraida"])
    if _local.exists():
        return _local
    if EN_COLAB:
        from google.colab import drive
        drive.mount("/content/drive")
        _dir, _zip = Path("/content/drive/MyDrive/base_sepa"), Path("/content/drive/MyDrive/base_sepa.zip")
        if _dir.exists():
            return _dir
        if _zip.exists() and not Path("/content/base_sepa").exists():
            import zipfile
            print("Extrayendo base_sepa.zip ...")
            with zipfile.ZipFile(_zip) as z:
                z.extractall("/content/base_sepa")
        return Path("/content/base_sepa")
    raise FileNotFoundError("No se encontro la base; configura la ruta.")

if not _hay_parquet():
    print("No hay Parquet. Construyo una muestra capeada (minorista + mayorista)...")
    archivos = descubrir_archivos(_localizar_base())
    a_min = next(a for a in archivos if a.tipo == "minorista")
    a_may = next(a for a in archivos if a.tipo == "mayorista")
    procesar_archivo(a_min, PARQUET, chunksize=100_000, limite_filas=300_000)
    procesar_archivo(a_may, PARQUET, chunksize=50_000,  limite_filas=200_000)

con = duckdb.connect()
con.execute(f"CREATE VIEW sepa AS SELECT * FROM read_parquet('{glob}', hive_partitioning=1, union_by_name=1)")
print("Filas totales:", con.sql("SELECT count(*) FROM sepa").fetchone()[0])
con.sql("SELECT tipo, anio, mes, count(*) filas FROM sepa GROUP BY 1,2,3 ORDER BY 2,3,1").df()

## 2. Maestros (productos y sucursales)
Reparados de mojibake y normalizados. Se anclan las sucursales a este maestro (no se pierde ninguna).

In [ ]:
from precios_sepa.maestros import cargar_productos, cargar_sucursales

APOYO = Path(cfg["rutas"]["base_extraida"]) / "Archivos_de_apoyo"
prod = cargar_productos(APOYO / "Maestro de Productos Interno.xlsx")
suc  = cargar_sucursales(APOYO / "maestro_sucursales_completo.xlsx")
print(f"Productos: {len(prod):,} | Sucursales: {len(suc):,}")
print("Rubros:", prod['rubro'].dropna().nunique(), "| Categorias:", prod['categoria'].dropna().nunique())
suc[['id_comercio','id_bandera','id_sucursal','sucursales_nombre','provincia','region']].head()

## 3. Universo de cadenas
`(id_comercio, id_bandera)` = cadena comercial. Se nombra con `config/cadenas.csv` y se etiqueta el resto sin descartar.

In [ ]:
from precios_sepa.cadenas import asignar_cadena

univ = con.sql("""
    SELECT id_comercio, id_bandera,
           count(DISTINCT id_sucursal) AS sucursales,
           count(DISTINCT id_producto) AS productos
    FROM sepa GROUP BY 1,2
""").df()
univ = asignar_cadena(univ).sort_values("sucursales", ascending=False)
print(f"{len(univ)} combinaciones (id_comercio,id_bandera) | "
      f"{univ['cadena'].str.startswith('Comercio ').sum()} sin identificar (fallback)")
univ.head(25)

## 4. Cobertura geográfica y por cadena

In [ ]:
import matplotlib.pyplot as plt

# Sucursales activas por provincia
cob_prov = con.sql("""
    SELECT sucursales_provincia AS iso, count(DISTINCT id_sucursal) AS sucursales,
           count(DISTINCT id_producto) AS productos
    FROM sepa GROUP BY 1 ORDER BY 2 DESC
""").df()
provs = pd.read_csv(REPO / "config" / "provincias.csv")
cob_prov = cob_prov.merge(provs, left_on="iso", right_on="iso_3166_2", how="left")

fig, ax = plt.subplots(1, 2, figsize=(13, 6))
d = cob_prov.dropna(subset=["provincia"]).sort_values("sucursales")
ax[0].barh(d["provincia"], d["sucursales"], color="#0055A4")
ax[0].set_title("Sucursales activas por provincia"); ax[0].tick_params(labelsize=8)

top = univ.head(15).sort_values("sucursales")
ax[1].barh(top["cadena"], top["sucursales"], color="#C1272D")
ax[1].set_title("Sucursales por cadena (top 15)"); ax[1].tick_params(labelsize=8)
plt.tight_layout(); plt.show()

## 5. Calidad de datos
Distribución de precios (tras limpiar sentinelas) y cobertura temporal.

In [ ]:
# Distribucion de precios por tipo (minorista vs mayorista uni_iva)
stats = con.sql("""
    SELECT tipo, COALESCE(tipo_precio,'(minorista)') AS medida,
           count(*) AS n,
           round(quantile_cont(precio, 0.25),1) AS p25,
           round(median(precio),1)              AS mediana,
           round(quantile_cont(precio, 0.75),1) AS p75,
           round(max(precio),1)                 AS maximo
    FROM sepa GROUP BY 1,2 ORDER BY 1,2
""").df()
stats

In [ ]:
# Histograma de precios (escala log) minorista
pm = con.sql("SELECT precio FROM sepa WHERE tipo='minorista' AND precio BETWEEN 1 AND 100000 USING SAMPLE 200000 ROWS").df()
if len(pm):
    plt.figure(figsize=(9,4))
    plt.hist(np.log10(pm["precio"]), bins=60, color="#0055A4", alpha=.85)
    plt.xlabel("log10(precio ARS)"); plt.ylabel("frecuencia")
    plt.title("Distribucion de precios minoristas (muestra, escala log)")
    plt.tight_layout(); plt.show()

## 6. Mapa de sucursales
Un punto por sucursal (lat/lon del maestro), coloreado por región.

In [ ]:
import folium

_COLOR_REGION = {"AMBA":"#0055A4","Pampeana":"#2ca02c","Cuyo":"#ff7f0e",
                 "Noroeste":"#9467bd","Noreste":"#8c564b","Patagonia":"#17becf"}
m = suc.dropna(subset=["sucursales_latitud","sucursales_longitud"]).copy()
m = m[(m["sucursales_latitud"].between(-56,-21)) & (m["sucursales_longitud"].between(-74,-53))]
mapa = folium.Map(location=[-38.4,-63.6], zoom_start=4, tiles="cartodbpositron")
for r in m.itertuples():
    folium.CircleMarker(
        [r.sucursales_latitud, r.sucursales_longitud], radius=2,
        color=_COLOR_REGION.get(getattr(r,'region',None), "#888"), fill=True, weight=0,
        popup=f"{getattr(r,'sucursales_nombre','')} ({getattr(r,'provincia','')})",
    ).add_to(mapa)
print(f"{len(m):,} sucursales en el mapa")
mapa

## 7. Guardar agregados livianos
Salidas reutilizables por los demás notebooks (livianas, publicables).

In [ ]:
PROC = REPO / "data" / "processed"; PROC.mkdir(parents=True, exist_ok=True)
OUTT = REPO / "outputs" / "tables"; OUTT.mkdir(parents=True, exist_ok=True)

univ.to_parquet(PROC / "universo_cadenas.parquet", index=False)
cob_prov.to_csv(OUTT / "cobertura_provincias.csv", index=False)
stats.to_csv(OUTT / "estadisticas_precio.csv", index=False)
try:
    mapa.save(str(REPO / "outputs" / "figures" / "mapa_sucursales.html"))
except Exception as e:
    print("mapa:", e)
print("Guardado en data/processed/ y outputs/.")

## Listo

Radiografía completa: cobertura, calidad, cadenas y mapa. Con esto, el siguiente paso es
**`02_construccion_canastas.ipynb`** (canasta típica + celíaca) y luego el
**`03_prima_celiaca.ipynb`** (Artículo 1).